# III-5 GPU を使い切るための基礎（ノートブック版）

教科書の III-5 の手順とコードを、上から順に実行すれば動く 1 冊にしたものです。**Google Colab の GPU（CUDA）で動かすこと**を主に考えています。Mac の Jupyter（MPS、または CPU）でも同じように動きます。

| 節 | 内容 | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | GPU と CPU のようすを見る（`nvidia-smi`、コア数） | 8 | ― |
| 2 | データを保存する（前処理の事前計算） | 2 | `prep_cache.py` |
| 3 | データ待ちを測る（合成・保存したデータ、ワーカー数） | 3 | `train_fast.py` |
| 4 | アムダールの法則で上限を出す | 4 | ― |
| 5 | 計算を速くする工夫（AMP・勾配累積・torch.compile）と表 | 5 | `train_fast.py`・`fast_table.py` |
| 6 | 前処理の置き場所（STFT・CWT を CPU／GPU／保存して読む） | 6 | `stft_bench.py` |
| 7 | プロファイラで 1 ステップの中身を見る | 7 | `prof_toy.py` |
| 8 | `gpu_log.sh` で GPU-Util を記録する（`nvidia-smi` があるときだけ） | 8・10 | `gpu_log.sh` |
| 9 | 確認問題の数値を確かめる、結果を持ち帰る | ― | ― |

- 次の「準備」のセルたちが、III-2 の `train_toy.py` と、この章の 6 ファイルを、このノートブックの隣に `%%writefile` で書き出します。**中身は教科書と 1 文字も変えていません**（ほかのファイルは要りません。この章のファイルが import するのは `train_toy.py` だけです）。
- 教科書のターミナルの `uv run --locked python train_fast.py --data synth --workers 0 --label synth_w0` の代わりに、`run("train_fast.py --data synth --workers 0 --label synth_w0 …")` で同じ引数を渡して別のプロセスで実行します（`run()` はこのノートブックで足した小さな関数です）。
- **既定は小さい設定**です（保存するデータ 4000 個、記録するステップ 40）。教科書の本番の設定（20000 個、120 ステップ。クラウドでは 300 ステップ）は、セルのコメントに書いてあります。時間の値は III-5-7 の表と合いません。`ok()` は、設定の大きさや機械に依らず成り立つこと（式の値、記録の形、表の列など）だけを確かめます。「ワーカーを増やすと待ちが減る」などの速さの比べは、`ok()` にせず表示だけにしています（機械で変わるため）。
- 執筆時の Mac（Apple シリコン、MPS）で全部で 5 分ほどです。
- **入れていないもの**：Vast.ai のインスタンスでの実行（手順 8〜10 の `vast_sync.sh`・`setup_uv_instance.sh`。このノートブックを Colab の GPU で動かすことが、その代わりの「クラウドで同じことをする」になります）、Git へのコミット（手順 1）。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

## 0. 準備

### Colab で GPU を選ぶ

1. メニューの「ランタイム」→「ランタイムのタイプを変更」を開き、ハードウェア アクセラレータで GPU（例：T4）を選んで保存します（項目の名前と、無料で選べる GPU の種類は公式ドキュメントで要確認）。
2. ランタイムを変えると、それまでに書き出したファイルと変数は消えるので、このノートブックは**最初のセルから**実行し直してください。
3. GPU を選ばなくても全部のセルは動きます（計算の場所が `cpu` になり、AMP は bf16 だけ、`gpu_log.sh` の節はとばします）。

計算の場所（device）は、III-2 の `pick_device()` が **cuda → mps → cpu** の順に自動で選びます。Colab の GPU では `cuda`、Mac（Apple シリコン）では `mps` です。

### Mac とクラウドで変える設定

教科書では、Mac は AMP なし（`--amp off`）、クラウドのインスタンスは III-3 の $\eta$ と比べられるよう `train_toy.py` と同じ fp16 の AMP を基本にしています（「インスタンス｜コマンドのまとめ」）。このノートブックも、`cuda` なら基本を `--amp fp16`、それ以外は `--amp off` にします。結果のフォルダも教科書と同じく、`cuda` なら `runs/fast_cuda`、それ以外は `runs/fast` です。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には torch・numpy・matplotlib が最初から入っている。版は公式で要確認）
import importlib, subprocess, sys
for mod, pkg in [("torch", "torch"), ("numpy", "numpy"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, torch, platform
print("Python", platform.python_version(), " numpy", numpy.__version__, " torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), " MPS:", torch.backends.mps.is_available())
print("（教科書は torch 2.14.1 で確かめた。Colab の torch の版は違うことがある）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

次のセルで III-2 の `train_toy.py` を書き出します。この章のファイルは、ここから `SynthMotor`（合成のモータ音）・`AudioCNN`（モデル）・`pick_device`・`device_name`・`set_seed`・`FS`（標本化周波数）を import します。

In [ ]:
%%writefile train_toy.py
#!/usr/bin/env python3
"""III-2：どこでも動く練習用の学習スクリプト
合成した「モータ音」の正常／異常を、1次元CNNで分類する。
- device を自動で選ぶ（cuda → mps → cpu）。同じコードが Mac と Colab で動く
- 一定時間ごと＋エポックの終わりにチェックポイントを保存。切断されても途中から正確に再開できる
- CUDA では混合精度（AMP）を使う
- --max-hours を過ぎたら、保存して自分で終了する（課金の止め忘れ対策）
使い方:
  uv run python train_toy.py --quick --ckpt-dir runs/mac_quick              # Mac で動作確認
  python train_toy.py --ckpt-dir /content/drive/MyDrive/home-lab/runs/toy01    # Colab で本番
"""
import argparse
import contextlib
import csv
import json
import math
import os
import platform
import random
import signal
import subprocess
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

FS = 16000
stop_requested = False


def on_signal(signum, frame):
    global stop_requested
    stop_requested = True
    print(f"\nシグナル {signum} を受信：区切りのよいところで保存して終了します", flush=True)


# ---------------------------------------------------------------- 環境
def pick_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def device_name(dev):
    if dev.type == "cuda":
        return torch.cuda.get_device_name(0)
    if dev.type == "mps":
        return "Apple MPS"
    return platform.processor() or platform.machine()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def git_commit():
    try:
        r = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                           capture_output=True, text=True, timeout=5)
        return r.stdout.strip() or "unknown"
    except Exception:
        return "unknown"


def amp_ctx(enabled):
    return torch.autocast("cuda", dtype=torch.float16) if enabled else contextlib.nullcontext()


# ---------------------------------------------------------------- データ（合成）
def make_signal(rng, length, anomalous):
    """基本波 80〜120 Hz と高調波＋雑音。異常は「周期的なクリック」か「高い周波数のうなり」"""
    t = np.arange(length) / FS
    f0 = rng.uniform(80, 120)
    x = np.zeros(length)
    for k in range(1, 6):
        x += np.sin(2 * np.pi * k * f0 * t + rng.uniform(0, 2 * np.pi)) / k
    x += rng.normal(0, 0.3, length)
    if anomalous:
        if rng.integers(2) == 0:
            period = int(FS / rng.uniform(15, 40))
            win = np.hanning(16) * rng.uniform(2, 4)
            for p in range(int(rng.integers(period)), length, period):
                seg = x[p:p + 16]
                seg += win[:len(seg)]
        else:
            fm = rng.uniform(1500, 3000)
            env = 1 + 0.5 * np.sin(2 * np.pi * rng.uniform(5, 20) * t)
            x += 0.6 * np.sin(2 * np.pi * fm * t) * env
    return (x / x.std()).astype(np.float32)


class SynthMotor(Dataset):
    """番号 i から決まる乱数で毎回同じ信号を作る（ファイルに保存しないのでディスクを使わない）"""
    def __init__(self, n, length, seed):
        self.n, self.length, self.seed = n, length, seed

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = np.random.default_rng([self.seed, i])
        y = i % 2                                 # 正常と異常を半分ずつ
        x = make_signal(rng, self.length, bool(y))
        return torch.from_numpy(x).unsqueeze(0), y


# ---------------------------------------------------------------- モデル
class AudioCNN(nn.Module):
    def __init__(self, w=32, ncls=2):
        super().__init__()
        layers, cin = [], 1
        for cout in (w, w, 2 * w, 2 * w, 4 * w, 4 * w):
            layers += [nn.Conv1d(cin, cout, 9, stride=2, padding=4),
                       nn.BatchNorm1d(cout), nn.ReLU(inplace=True)]
            cin = cout
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(cin, ncls)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=-1))


# ---------------------------------------------------------------- チェックポイント
def save_ckpt(path, state):
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)                         # 書き込み途中で切れても壊れない


@torch.no_grad()
def evaluate(model, loader, dev, use_amp):
    model.eval()
    correct = n = 0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        with amp_ctx(use_amp):
            pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        n += len(y)
    model.train()
    return correct / n


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--n-train", type=int, default=20000)
    ap.add_argument("--n-val", type=int, default=2000)
    ap.add_argument("--length", type=int, default=4096)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--no-amp", action="store_true")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--ckpt-dir", default="runs/toy")
    ap.add_argument("--ckpt-every-min", type=float, default=5.0)
    ap.add_argument("--max-hours", type=float, default=3.0, help="この時間を過ぎたら保存して終了")
    ap.add_argument("--quick", action="store_true", help="小さい設定で動作確認")
    a = ap.parse_args()
    if a.quick:
        a.n_train, a.n_val, a.epochs = 4000, 400, 2
    return a


def main():
    a = parse()
    signal.signal(signal.SIGTERM, on_signal)
    signal.signal(signal.SIGINT, on_signal)
    set_seed(a.seed)
    dev = pick_device(a.device)
    use_amp = dev.type == "cuda" and not a.no_amp
    if dev.type == "cuda":
        torch.backends.cudnn.benchmark = True
    os.makedirs(a.ckpt_dir, exist_ok=True)
    last = os.path.join(a.ckpt_dir, "last.pt")
    metrics_path = os.path.join(a.ckpt_dir, "metrics.csv")
    t_start = time.time()

    train_ds = SynthMotor(a.n_train, a.length, seed=a.seed)
    val_ds = SynthMotor(a.n_val, a.length, seed=a.seed + 10_000)
    pin = dev.type == "cuda"
    val_loader = DataLoader(val_ds, batch_size=a.batch * 2, num_workers=a.workers, pin_memory=pin)

    model = AudioCNN().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    start_epoch, start_step = 0, 0
    run = {"loss_sum": 0.0, "n": 0, "t": 0.0}
    resumed = False
    if os.path.exists(last):
        ck = torch.load(last, map_location="cpu", weights_only=False)   # 自分で作ったファイルなので可
        if ck.get("done"):
            print(f"この実行はすでに完了しています: {last}")
            return
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        if use_amp and ck.get("scaler"):
            scaler.load_state_dict(ck["scaler"])
        start_epoch, start_step, run = ck["epoch"], ck["step"], ck["run"]
        resumed = True
        print(f"チェックポイントから再開: epoch {start_epoch} step {start_step}", flush=True)

    # 実行の記録（再開のたびに追記）
    info_path = os.path.join(a.ckpt_dir, "run_info.json")
    info = json.load(open(info_path, encoding="utf-8")) if os.path.exists(info_path) else {"sessions": []}
    info["sessions"].append({
        "started": time.strftime("%Y-%m-%dT%H:%M:%S%z"), "git_commit": git_commit(),
        "torch": torch.__version__, "python": sys.version.split()[0],
        "device": dev.type, "device_name": device_name(dev), "amp": use_amp,
        "args": vars(a), "resumed": resumed})
    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)
    print(f"device = {device_name(dev)}（{dev.type}）, AMP = {use_amp}, commit = {git_commit()}", flush=True)

    def state(epoch, step, run_state, done=False):
        return {"model": model.state_dict(), "opt": opt.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch, "step": step,
                "run": run_state, "args": vars(a), "done": done,
                "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

    last_save = time.time()
    for epoch in range(start_epoch, a.epochs):
        first = epoch == start_epoch
        if not first:
            run = {"loss_sum": 0.0, "n": 0, "t": 0.0}
        step = start_step if first else 0
        # エポックごとの並び順を「種＋エポック番号」で決める → 途中からでも同じ順番で再開できる
        g = torch.Generator().manual_seed(a.seed * 1000 + epoch)
        order = torch.randperm(len(train_ds), generator=g).tolist()
        loader = DataLoader(Subset(train_ds, order[step * a.batch:]), batch_size=a.batch,
                            shuffle=False, num_workers=a.workers, pin_memory=pin)
        model.train()
        t_seg = time.time()
        for x, y in loader:
            x = x.to(dev, non_blocking=True)
            y = y.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with amp_ctx(use_amp):
                loss = F.cross_entropy(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run["loss_sum"] += loss.item() * len(y)
            run["n"] += len(y)
            step += 1
            over_time = time.time() - t_start > a.max_hours * 3600
            if stop_requested or over_time or time.time() - last_save > a.ckpt_every_min * 60:
                run["t"] += time.time() - t_seg
                t_seg = time.time()
                save_ckpt(last, state(epoch, step, run))
                last_save = time.time()
                print(f"  チェックポイント保存: epoch {epoch} step {step}", flush=True)
                if stop_requested or over_time:
                    why = "シグナル" if stop_requested else f"--max-hours {a.max_hours}"
                    print(f"{why} により終了します。同じコマンドで再開できます。", flush=True)
                    return
        run["t"] += time.time() - t_seg
        val_acc = evaluate(model, val_loader, dev, use_amp)
        row = {"epoch": epoch, "train_loss": run["loss_sum"] / max(run["n"], 1),
               "val_acc": val_acc, "epoch_time_s": run["t"],
               "samples_per_s": run["n"] / max(run["t"], 1e-9),
               "device": dev.type, "device_name": device_name(dev),
               "resumed": resumed and first, "time": time.strftime("%Y-%m-%dT%H:%M:%S%z")}
        new = not os.path.exists(metrics_path)
        with open(metrics_path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row.keys()))
            if new:
                w.writeheader()
            w.writerow(row)
        print(f"epoch {epoch}: loss {row['train_loss']:.4f}  val_acc {val_acc:.4f}  "
              f"{row['samples_per_s']:.0f} サンプル/秒", flush=True)
        save_ckpt(last, state(epoch + 1, 0, {"loss_sum": 0.0, "n": 0, "t": 0.0},
                              done=(epoch + 1 == a.epochs)))
        last_save = time.time()
    print("完了しました。", flush=True)


if __name__ == "__main__":     # DataLoader の並列読み込み（macOS の spawn）に必要
    main()

次の 6 つのセルで、この章のファイルを書き出します（教科書 III-5-6 と同じもの）。`train_fast.py` の `if __name__ == "__main__":` は、DataLoader のワーカー（macOS の spawn）に必要です。

In [ ]:
%%writefile gpu_log.sh
#!/usr/bin/env bash
# III-5：コマンドを実行している間、nvidia-smi で GPU の状態を1秒ごとに CSV に記録する
# 使い方: bash gpu_log.sh <名前> -- <実行するコマンド…>
#   例: bash gpu_log.sh synth_w0 -- uv run --locked python train_fast.py --data synth --workers 0 --label synth_w0
# 出力: ${GPU_LOG_DIR:-runs/gpu_log}/<名前>.csv（nvidia-smi が無い Mac では記録せず、コマンドだけ実行する）
# nvidia-smi は CUDA_VISIBLE_DEVICES を見ずに全部の GPU を表示する。共有サーバー（III-7）で
# ほかの人の GPU が混ざらないよう、CUDA_VISIBLE_DEVICES があればその GPU だけを -i で記録し、
# まとめも GPU の番号（index）ごとに出す
set -euo pipefail
name="${1:?名前を指定してください}"
shift
[ "${1:-}" = "--" ] && shift
[ $# -gt 0 ] || { echo "実行するコマンドを -- のあとに書いてください" >&2; exit 2; }

if ! command -v nvidia-smi >/dev/null 2>&1; then
  echo "nvidia-smi がありません（Mac など）。GPU の記録はせずにコマンドだけ実行します。" >&2
  exec "$@"
fi

dir="${GPU_LOG_DIR:-runs/gpu_log}"
mkdir -p "${dir}"
out="${dir}/${name}.csv"
fields="timestamp,index,name,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw,power.limit,temperature.gpu,clocks.sm"

# 記録する GPU を選ぶ（CUDA_VISIBLE_DEVICES が無い・空なら全部）
sel=()
cvd="${CUDA_VISIBLE_DEVICES:-}"
if [ -n "${cvd}" ]; then
  if nvidia-smi -i "${cvd}" -L >/dev/null 2>&1; then
    sel=(-i "${cvd}")
    echo "記録する GPU：CUDA_VISIBLE_DEVICES=${cvd}（nvidia-smi の番号として -i に渡す）" >&2
    case "${cvd}" in
      *GPU-*|*MIG-*) ;;                       # UUID なら並び順の問題は無い
      *) [ "${CUDA_DEVICE_ORDER:-}" = "PCI_BUS_ID" ] ||
           echo "注意：CUDA_DEVICE_ORDER=PCI_BUS_ID が無いと、CUDA の番号と nvidia-smi の番号がずれることがあります" >&2 ;;
    esac
  else
    echo "注意：nvidia-smi -i ${cvd} が通らないので全部の GPU を記録します。まとめの GPU ごとの行から自分の GPU を選んでください" >&2
  fi
fi
: > "${out}"                                   # 前の記録を先に消しておく
nvidia-smi "${sel[@]+"${sel[@]}"}" --query-gpu="${fields}" --format=csv,nounits -l 1 > "${out}" &
logger=$!
trap 'kill "${logger}" 2>/dev/null || true' EXIT

status=0
"$@" || status=$?

kill "${logger}" 2>/dev/null || true
wait "${logger}" 2>/dev/null || true
trap - EXIT

# まとめ：GPU ごとに、GPU-Util の平均と、50% 未満だった秒の割合、メモリと電力の最大
awk -F', *' 'NR > 1 && $4 ~ /^[0-9.]+$/ {
  g = $2; if (!(g in n)) order[++ng] = g
  n[g]++; u[g] += $4; if ($4 < 50) low[g]++
  if ($6 > m[g]) m[g] = $6; tot[g] = $7
  if ($8 ~ /^[0-9.]+$/ && $8 > p[g]) p[g] = $8; lim[g] = $9
  if ($10 > tc[g]) tc[g] = $10
} END {
  if (ng == 0) { print "記録がありません"; exit }
  for (k = 1; k <= ng; k++) {
    g = order[k]
    printf "GPU %s：記録 %d 秒、GPU-Util 平均 %.0f%%、50%% 未満の秒 %.0f%%、メモリ最大 %d / %d MiB、電力最大 %.0f / %s W、温度最大 %d ℃\n",
           g, n[g], u[g] / n[g], 100 * low[g] / n[g], m[g], tot[g], p[g], lim[g], tc[g]
  }
}' "${out}"
echo "記録：${out}"
exit "${status}"

In [ ]:
%%writefile prep_cache.py
#!/usr/bin/env python3
"""III-5：train_toy.py の合成データを、前もって計算してファイルに保存する（前処理の事前計算）
- 波形を data/<名前>/x.npy（float32、形 (N, L)）、ラベルを y.npy に保存する
- --stft を付けると、対数振幅スペクトログラム（float16、形 (N, F, T)）も stft.npy に保存する
- 学習のときは CachedMotor で np.load(mmap_mode="r") して読む（メモリに全部は載せない）
使い方:
  uv run python prep_cache.py --n 20000 --length 4096 --seed 0 --name toy_train
  uv run python prep_cache.py --n 20000 --length 4096 --seed 0 --name toy_train --stft
"""
import argparse
import json
import os
import time

import numpy as np
import torch
from torch.utils.data import Dataset

from train_toy import SynthMotor

N_FFT, HOP = 256, 64


def log_stft(x, n_fft=N_FFT, hop=HOP):
    """x：(B, L) の波形 → (B, n_fft/2+1, フレーム数) の log(1+|X|)。CPU でも GPU でも同じ関数"""
    win = torch.hann_window(n_fft, device=x.device)
    X = torch.stft(x, n_fft=n_fft, hop_length=hop, window=win, center=True, return_complex=True)
    return torch.log1p(X.abs())


class CachedMotor(Dataset):
    """prep_cache.py が作ったファイルを読む。SynthMotor と同じく (x, y) を返す
    kind="wave" なら (1, L) の波形、kind="stft" なら (F, T) のスペクトログラム"""
    def __init__(self, folder, kind="wave"):
        self.folder, self.kind = folder, kind
        self.y = np.load(os.path.join(folder, "y.npy"))
        self.x = None                     # 開くのは各ワーカーの中（fork・spawn どちらでも安全）

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        if self.x is None:
            name = "x.npy" if self.kind == "wave" else "stft.npy"
            self.x = np.load(os.path.join(self.folder, name), mmap_mode="r")
        a = torch.from_numpy(np.array(self.x[i], dtype=np.float32))   # コピーしてから渡す
        if self.kind == "wave":
            a = a.unsqueeze(0)
        return a, int(self.y[i])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=20000)
    ap.add_argument("--length", type=int, default=4096)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--name", default="toy_train")
    ap.add_argument("--root", default="data")
    ap.add_argument("--stft", action="store_true", help="スペクトログラムも保存する")
    ap.add_argument("--chunk", type=int, default=512)
    a = ap.parse_args()

    out = os.path.join(a.root, a.name)
    os.makedirs(out, exist_ok=True)
    ds = SynthMotor(a.n, a.length, seed=a.seed)
    t0 = time.perf_counter()
    x = np.lib.format.open_memmap(os.path.join(out, "x.npy"), mode="w+",
                                  dtype=np.float32, shape=(a.n, a.length))
    y = np.zeros(a.n, dtype=np.int64)
    for i in range(a.n):
        xi, yi = ds[i]
        x[i] = xi[0].numpy()
        y[i] = yi
    x.flush()
    np.save(os.path.join(out, "y.npy"), y)
    t_wave = time.perf_counter() - t0
    print(f"波形 {a.n} 個を保存：{t_wave:.1f} 秒（{a.n / t_wave:.0f} 個/秒）、"
          f"{x.nbytes / 1e6:.0f} MB → {out}/x.npy", flush=True)

    info = {"n": a.n, "length": a.length, "seed": a.seed, "source": "train_toy.SynthMotor",
            "wave_s": round(t_wave, 2)}
    if a.stft:
        t0 = time.perf_counter()
        F_ = log_stft(torch.zeros(1, a.length)).shape[1:]
        s = np.lib.format.open_memmap(os.path.join(out, "stft.npy"), mode="w+",
                                      dtype=np.float16, shape=(a.n, *F_))
        for k in range(0, a.n, a.chunk):
            xb = torch.from_numpy(np.asarray(x[k:k + a.chunk]))
            s[k:k + len(xb)] = log_stft(xb).numpy().astype(np.float16)
        s.flush()
        t_stft = time.perf_counter() - t0
        info.update({"stft_shape": list(F_), "n_fft": N_FFT, "hop": HOP, "stft_s": round(t_stft, 2)})
        print(f"STFT（CPU）：{t_stft:.1f} 秒、形 {tuple(F_)}、{s.nbytes / 1e6:.0f} MB → {out}/stft.npy")
    with open(os.path.join(out, "cache_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train_fast.py
#!/usr/bin/env python3
"""III-5：GPU がデータを待っている時間を測り、速くする工夫を1つずつ試す
train_toy.py と同じデータ（SynthMotor）・同じモデル（AudioCNN）で、決めたステップ数だけ学習し、
1ステップを「データ待ち」と「GPU の計算」に分けて記録する。
- --data synth：train_toy.py と同じく、信号を毎回 CPU で作る
- --data cache：prep_cache.py で保存したファイル（memmap）を読む
- --amp off|fp16|bf16、--accum K（勾配累積）、--compile（torch.compile）
使い方:
  uv run python train_fast.py --data synth --workers 0 --label synth_w0
  uv run python train_fast.py --data cache --cache-dir data/toy_train --workers 2 --label cache_w2
"""
import argparse
import contextlib
import csv
import json
import os
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from prep_cache import CachedMotor
from train_toy import AudioCNN, SynthMotor, device_name, pick_device, set_seed


def sync(dev):
    """GPU の計算は非同期なので、時刻を測る前に終わるのを待つ"""
    if dev.type == "cuda":
        torch.cuda.synchronize()
    elif dev.type == "mps":
        torch.mps.synchronize()


def autocast_ctx(dev, amp):
    if amp == "off":
        return contextlib.nullcontext()
    dtype = torch.float16 if amp == "fp16" else torch.bfloat16
    return torch.autocast(dev.type, dtype=dtype)


def peak_mem_mb(dev):
    if dev.type == "cuda":
        return torch.cuda.max_memory_allocated() / 2**20
    if dev.type == "mps":
        return torch.mps.driver_allocated_memory() / 2**20   # MPS は最大値ではなく今の確保量
    return float("nan")


def amdahl(f, s):
    """速くなる部分の割合 f を s 倍にしたときの全体の倍率"""
    return 1.0 / ((1.0 - f) + f / s)


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", choices=["synth", "cache"], default="synth")
    ap.add_argument("--cache-dir", default="data/toy_train")
    ap.add_argument("--n-train", type=int, default=20000)
    ap.add_argument("--length", type=int, default=4096)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--accum", type=int, default=1, help="この回数ぶん勾配をためてから更新する")
    ap.add_argument("--workers", type=int, default=0)
    ap.add_argument("--prefetch", type=int, default=2, help="ワーカー1つが先に用意するバッチ数")
    ap.add_argument("--amp", choices=["off", "fp16", "bf16"], default="off")
    ap.add_argument("--compile", action="store_true")
    ap.add_argument("--steps", type=int, default=120, help="記録するステップ数（ウォームアップは別）")
    ap.add_argument("--warmup", type=int, default=10)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--label", default="run")
    ap.add_argument("--outdir", default="runs/fast")
    return ap.parse_args()


def main():
    a = parse()
    set_seed(a.seed)
    dev = pick_device(a.device)
    if a.amp == "fp16" and dev.type == "cpu":
        raise SystemExit("CPU では --amp bf16 を使ってください（fp16 の autocast は遅いか使えない）")
    if dev.type == "cuda":
        torch.backends.cudnn.benchmark = True

    if a.data == "synth":
        ds = SynthMotor(a.n_train, a.length, seed=a.seed)
    else:
        ds = CachedMotor(a.cache_dir, kind="wave")
    pin = dev.type == "cuda"
    kw = dict(batch_size=a.batch, shuffle=True, num_workers=a.workers, pin_memory=pin,
              drop_last=True, generator=torch.Generator().manual_seed(a.seed))
    if a.workers > 0:
        kw.update(persistent_workers=True, prefetch_factor=a.prefetch)
    loader = DataLoader(ds, **kw)

    model = AudioCNN().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr)
    use_scaler = dev.type == "cuda" and a.amp == "fp16"
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    step_model = torch.compile(model) if a.compile else model

    total = a.warmup + a.steps
    rows, it, n_micro = [], None, total * a.accum
    if dev.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t_begin = time.perf_counter()
    sync(dev)
    t_prev = time.perf_counter()
    for k in range(total):
        wait = comp = 0.0
        opt.zero_grad(set_to_none=True)
        for m in range(a.accum):
            if it is None:
                it = iter(loader)
            try:
                x, y = next(it)
            except StopIteration:              # エポックの終わり：もう一周
                it = iter(loader)
                x, y = next(it)
            x = x.to(dev, non_blocking=True)
            y = y.to(dev, non_blocking=True)
            sync(dev)
            t_data = time.perf_counter()
            wait += t_data - t_prev
            with autocast_ctx(dev, a.amp):
                loss = F.cross_entropy(step_model(x), y) / a.accum
            scaler.scale(loss).backward()
            if m == a.accum - 1:
                scaler.step(opt)
                scaler.update()
            sync(dev)
            t_prev = time.perf_counter()
            comp += t_prev - t_data
        rows.append({"step": k, "warmup": k < a.warmup, "wait_s": wait, "compute_s": comp,
                     "loss": loss.item() * a.accum})
    wall = time.perf_counter() - t_begin

    rec = [r for r in rows if not r["warmup"]]
    w = np.array([r["wait_s"] for r in rec])
    c = np.array([r["compute_s"] for r in rec])
    step_s = w + c
    f_data = w.sum() / step_s.sum()
    samples = a.batch * a.accum
    summ = {
        "label": a.label, "device": dev.type, "device_name": device_name(dev),
        "torch": torch.__version__, "data": a.data, "workers": a.workers, "batch": a.batch,
        "accum": a.accum, "amp": a.amp, "compile": a.compile, "steps": a.steps,
        "samples_per_s": samples * len(rec) / step_s.sum(),
        "step_ms_median": 1000 * float(np.median(step_s)),
        "wait_ms_median": 1000 * float(np.median(w)),
        "compute_ms_median": 1000 * float(np.median(c)),
        "f_data": float(f_data),
        "first_step_s": rows[0]["wait_s"] + rows[0]["compute_s"],
        "wall_s": wall, "peak_mem_mb": peak_mem_mb(dev),
        "final_loss": float(np.mean([r["loss"] for r in rec[-20:]])),
    }
    # アムダールの法則：データ待ち以外（GPU の計算）だけが速くなる場合の上限
    f_gpu = 1 - f_data
    summ["amdahl_x2"] = amdahl(f_gpu, 2)
    summ["amdahl_limit"] = 1 / f_data if f_data > 0 else float("inf")

    os.makedirs(a.outdir, exist_ok=True)
    with open(os.path.join(a.outdir, f"{a.label}_steps.csv"), "w", newline="") as fh:
        wr = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        wr.writeheader()
        wr.writerows(rows)
    with open(os.path.join(a.outdir, f"{a.label}.json"), "w", encoding="utf-8") as fh:
        json.dump(summ, fh, ensure_ascii=False, indent=2)

    print(f"[{a.label}] {summ['device_name']}（{dev.type}） data={a.data} workers={a.workers} "
          f"batch={a.batch}×{a.accum} amp={a.amp} compile={a.compile}")
    print(f"  {summ['samples_per_s']:.0f} サンプル/秒、1ステップ 中央値 {summ['step_ms_median']:.1f} ms"
          f"（データ待ち {summ['wait_ms_median']:.1f} ms ＋ 計算 {summ['compute_ms_median']:.1f} ms）")
    print(f"  データ待ちの割合 {100 * f_data:.0f}%  → GPU を2倍速くしても全体は "
          f"{summ['amdahl_x2']:.2f} 倍、無限に速くしても {summ['amdahl_limit']:.1f} 倍まで")
    print(f"  最初のステップ {summ['first_step_s']:.2f} 秒、メモリ {summ['peak_mem_mb']:.0f} MB、"
          f"損失（最後の20ステップの平均）{summ['final_loss']:.3f}")


if __name__ == "__main__":     # DataLoader のワーカー（macOS の spawn）に必要
    main()

In [ ]:
%%writefile fast_table.py
#!/usr/bin/env python3
"""III-5：train_fast.py の結果（runs/fast/*.json）を1つの表にし、基準との倍率とアムダールの上限を出す
使い方:
  uv run python fast_table.py                 # 基準は synth_w0
  uv run python fast_table.py --base cache_w2 --dir runs/fast_cuda
"""
import argparse
import glob
import json
import os


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dir", default="runs/fast")
    ap.add_argument("--base", default="synth_w0")
    a = ap.parse_args()
    rs = []
    for p in sorted(glob.glob(os.path.join(a.dir, "*.json"))):
        r = json.load(open(p, encoding="utf-8"))
        if "samples_per_s" in r:              # stft_bench.json などはとばす
            rs.append(r)
    if not rs:
        raise SystemExit(f"{a.dir} に train_fast.py の結果がありません")
    base = next((r for r in rs if r["label"] == a.base), rs[0])
    print(f"device: {base['device_name']}（{base['device']}）、torch {base['torch']}、基準: {base['label']}")
    print(f"{'label':<18}{'data':<6}{'wk':>3}{'batch':>8}{'amp':>5}{'comp':>5}"
          f"{'サンプル/秒':>10}{'倍率':>6}{'待ち%':>6}{'待ちms':>7}{'計算ms':>7}{'初回s':>6}{'上限':>6}")
    for r in sorted(rs, key=lambda r: r["samples_per_s"]):
        bt = f"{r['batch']}x{r['accum']}"
        print(f"{r['label']:<18}{r['data']:<6}{r['workers']:>3}{bt:>8}{r['amp']:>5}"
              f"{'yes' if r['compile'] else 'no':>5}{r['samples_per_s']:>12.0f}"
              f"{r['samples_per_s'] / base['samples_per_s']:>7.2f}{100 * r['f_data']:>7.0f}"
              f"{r['wait_ms_median']:>8.1f}{r['compute_ms_median']:>8.1f}{r['first_step_s']:>7.2f}"
              f"{min(r['amdahl_limit'], 999):>7.1f}")
    print("上限：GPU の計算を無限に速くしたときの全体の倍率 1/(データ待ちの割合)。"
          "計算ms と 待ちms を比べ、大きいほうから手を付ける")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile prof_toy.py
#!/usr/bin/env python3
"""III-5：torch.profiler で、train_toy.py の1ステップの中身（どの演算に時間がかかるか）を見る
CUDA では GPU の時間も、MPS・CPU では CPU 側の時間を集計する。結果の表を runs/fast/prof_<device>.txt に、
CUDA のときは Chrome のトレース（chrome://tracing や https://ui.perfetto.dev で開く）も保存する。
使い方:
  uv run python prof_toy.py --workers 0
"""
import argparse
import os

import torch
import torch.nn.functional as F
from torch.profiler import ProfilerActivity, profile, record_function, schedule
from torch.utils.data import DataLoader

from train_fast import sync
from train_toy import AudioCNN, SynthMotor, pick_device


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--workers", type=int, default=0)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--outdir", default="runs/fast")
    a = ap.parse_args()
    dev = pick_device("auto")
    loader = DataLoader(SynthMotor(4000, 4096, seed=0), batch_size=a.batch, shuffle=True,
                        num_workers=a.workers, drop_last=True)
    model = AudioCNN().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    acts = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if dev.type == "cuda" else [])
    # 最初の2ステップはとばし、2ステップ温めてから、6ステップを記録する
    sched = schedule(skip_first=2, wait=0, warmup=2, active=6, repeat=1)
    it = iter(loader)
    with profile(activities=acts, schedule=sched, record_shapes=False) as prof:
        for _ in range(10):
            with record_function("data_wait"):
                x, y = next(it)
                x, y = x.to(dev), y.to(dev)
            with record_function("train_step"):
                opt.zero_grad(set_to_none=True)
                loss = F.cross_entropy(model(x), y)
                loss.backward()
                opt.step()
                sync(dev)
            prof.step()
    key = "self_device_time_total" if dev.type == "cuda" else "self_cpu_time_total"
    table = prof.key_averages().table(sort_by=key, row_limit=15)
    print(table)
    os.makedirs(a.outdir, exist_ok=True)
    with open(os.path.join(a.outdir, f"prof_{dev.type}.txt"), "w") as f:
        f.write(table)
    if dev.type == "cuda":
        prof.export_chrome_trace(os.path.join(a.outdir, "prof_cuda_trace.json"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile stft_bench.py
#!/usr/bin/env python3
"""III-5：前処理（STFT・CWT）を「CPU で毎回」「GPU で毎回」「前もって計算して読むだけ」で比べる
1バッチ（--batch 本の波形）あたりの時間を、ウォームアップのあと --reps 回測って中央値を出す。
CWT は複素モルレーのウェーブレットのフィルタを並べ、conv1d（畳み込み）として計算する。
使い方（先に prep_cache.py --stft でファイルを作っておく）:
  uv run python stft_bench.py --cache-dir data/toy_train
"""
import argparse
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn.functional as F

from prep_cache import HOP, N_FFT, log_stft
from train_fast import sync
from train_toy import FS, pick_device


def morlet_bank(freqs, fs=FS, n_cycles=6.0, device="cpu"):
    """中心周波数 freqs の複素モルレーのフィルタ（実部・虚部を別の出力チャネルに）"""
    half = int(  # いちばん低い周波数のガウス窓の ±3σ が入る長さ
        math.ceil(3 * n_cycles / (2 * math.pi * freqs.min().item()) * fs))
    t = torch.arange(-half, half + 1, dtype=torch.float32) / fs
    w = []
    for f in freqs.tolist():
        s = n_cycles / (2 * math.pi * f)                               # ガウス窓の標準偏差 [s]
        g = torch.exp(-0.5 * (t / s) ** 2)
        g = g / g.sum()
        w += [g * torch.cos(2 * math.pi * f * t), g * torch.sin(2 * math.pi * f * t)]
    return torch.stack(w).unsqueeze(1).to(device)                      # (2K, 1, 長さ)


def cwt_mag(x, bank, hop=HOP):
    """x：(B, L) → (B, K, L/hop) の |W|。stride=hop で間引いて STFT と同じくらいの大きさにする"""
    y = F.conv1d(x.unsqueeze(1), bank, stride=hop, padding=bank.shape[-1] // 2)
    re, im = y[:, 0::2], y[:, 1::2]
    return torch.sqrt(re ** 2 + im ** 2)


def bench(fn, dev, reps, warmup=3):
    for _ in range(warmup):
        fn()
    sync(dev)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        sync(dev)
        ts.append(time.perf_counter() - t0)
    return 1000 * float(np.median(ts))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache-dir", default="data/toy_train")
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--reps", type=int, default=30)
    ap.add_argument("--n-freqs", type=int, default=48)
    ap.add_argument("--out", default="runs/fast/stft_bench.json")
    a = ap.parse_args()
    dev = pick_device("auto")
    rng = np.random.default_rng(0)
    xw = np.load(os.path.join(a.cache_dir, "x.npy"), mmap_mode="r")
    xs = np.load(os.path.join(a.cache_dir, "stft.npy"), mmap_mode="r")
    n = len(xw)
    idx = lambda: np.sort(rng.choice(n, a.batch, replace=False))     # 毎回ちがう番号を読む

    xb = torch.from_numpy(np.array(xw[: a.batch]))
    xg = xb.to(dev)
    freqs = torch.logspace(math.log10(50), math.log10(6000), a.n_freqs)
    bank_c, bank_g = morlet_bank(freqs), morlet_bank(freqs, device=dev)

    res = {"device": dev.type, "batch": a.batch, "n_fft": N_FFT, "hop": HOP,
           "cwt_freqs": a.n_freqs, "cwt_taps": bank_c.shape[-1]}
    res["stft_cpu_ms"] = bench(lambda: log_stft(xb), torch.device("cpu"), a.reps)
    res["stft_gpu_ms"] = bench(lambda: log_stft(xb.to(dev)).cpu(), dev, a.reps)      # 転送の往復込み
    res["stft_gpu_only_ms"] = bench(lambda: log_stft(xg), dev, a.reps)               # 載せたまま
    res["load_stft_ms"] = bench(lambda: torch.from_numpy(np.asarray(xs[idx()], dtype=np.float32)),
                                torch.device("cpu"), a.reps)
    res["load_wave_ms"] = bench(lambda: torch.from_numpy(np.array(xw[idx()])),
                                torch.device("cpu"), a.reps)
    res["cwt_cpu_ms"] = bench(lambda: cwt_mag(xb, bank_c), torch.device("cpu"), max(5, a.reps // 3))
    res["cwt_gpu_ms"] = bench(lambda: cwt_mag(xg, bank_g), dev, a.reps)
    # 答え合わせ：CPU と GPU の結果が（float32 の誤差の範囲で）同じか
    s1, s2 = log_stft(xb), log_stft(xg).cpu()
    c1, c2 = cwt_mag(xb, bank_c), cwt_mag(xg, bank_g).cpu()
    res["stft_max_rel_err"] = float((s1 - s2).abs().max() / s1.abs().max())
    res["cwt_max_rel_err"] = float((c1 - c2).abs().max() / c1.abs().max())
    res["cwt_shape"] = list(c1.shape[1:])
    res["stft_shape"] = list(s1.shape[1:])

    os.makedirs(os.path.dirname(a.out), exist_ok=True)
    with open(a.out, "w", encoding="utf-8") as f:
        json.dump(res, f, ensure_ascii=False, indent=2)
    print(f"device = {dev.type}、1バッチ = {a.batch} 本 × {xb.shape[1]} 点")
    print(f"  STFT  CPU {res['stft_cpu_ms']:7.2f} ms ｜ GPU（転送込み）{res['stft_gpu_ms']:7.2f} ms ｜"
          f" GPU（載せたまま）{res['stft_gpu_only_ms']:7.2f} ms  形 {tuple(s1.shape[1:])}")
    print(f"  CWT   CPU {res['cwt_cpu_ms']:7.2f} ms ｜ GPU（載せたまま）{res['cwt_gpu_ms']:7.2f} ms"
          f"  形 {tuple(c1.shape[1:])}、フィルタ {bank_c.shape[-1]} タップ")
    print(f"  読むだけ  保存した STFT {res['load_stft_ms']:6.2f} ms ｜ 保存した波形 {res['load_wave_ms']:6.2f} ms")
    print(f"  CPU と GPU の最大相対誤差：STFT {res['stft_max_rel_err']:.1e}、CWT {res['cwt_max_rel_err']:.1e}")


if __name__ == "__main__":
    main()

In [ ]:
# device・結果のフォルダ・基本の AMP を決め、run()・ok()・load() を用意する（ノートブック版で足したもの）
import os, sys, json, math, shlex, shutil, subprocess
import numpy as np
import torch
sys.path.insert(0, ".")
from train_toy import pick_device, device_name
from train_fast import amdahl

IN_COLAB = "google.colab" in sys.modules
os.environ["GIT_CEILING_DIRECTORIES"] = os.path.dirname(os.path.abspath("."))   # train_toy.py の git_commit() が、このフォルダより上の Git のリポジトリ（教科書のフォルダなど）のコミット ID を拾わないように
DEV = pick_device()
R = "runs/fast_cuda" if DEV.type == "cuda" else "runs/fast"        # 教科書と同じ置き場所
BASE_AMP = "fp16" if DEV.type == "cuda" else "off"                   # クラウドは fp16 が基本（III-3 の η と比べるため）
NCPU = os.cpu_count()
W_FAST = 2                       # (2) の基準のワーカー数（教科書：Mac は 2、インスタンスは 4）
WORKERS = [w for w in (0, 2, 4, 8) if w == 0 or w <= NCPU][:3]      # (1) で試すワーカー数（コア数まで。教科書：Mac は 0 2 4、インスタンスは 0 2 4 8）
STEPS, WARMUP = 40, 10           # 記録するステップ数（教科書の本番：Mac は 120、インスタンスは 300）
N_CACHE = 4000                   # 保存するデータの数（教科書の本番：20000）
os.environ["TORCHINDUCTOR_CACHE_DIR"] = os.path.abspath(".cache/inductor")   # 教科書の export TORCHINDUCTOR_CACHE_DIR=…
os.makedirs(R, exist_ok=True)
print(f"device {DEV}（{device_name(DEV)}）、基本の AMP {BASE_AMP}、結果は {R}/、CPU のコア {NCPU} 個、試すワーカー数 {WORKERS}")


def run(cmd, prefix=()):
    """ターミナルの `uv run --locked python <cmd>` の代わり。別のプロセスで実行し、出力をそのまま表示して返す
    prefix に ("bash", "gpu_log.sh", "名前", "--") を渡すと、教科書の `bash gpu_log.sh 名前 -- uv run … python …` と同じ"""
    args = [*prefix, sys.executable, "-u", *shlex.split(cmd)]
    print("$", " ".join(prefix), "python", cmd, flush=True)
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=dict(os.environ))
    lines = []
    for line in p.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    rc = p.wait()
    assert rc == 0, f"エラーで終わりました（returncode {rc}）。上の表示を見てください"
    return "".join(lines)


N_OK = 0
def ok(cond, msg):
    global N_OK
    assert cond, "★確かめ失敗：" + msg
    N_OK += 1
    print("OK", msg)


def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def check_run(label):
    """train_fast.py の結果（json と steps.csv）が、本文の定義どおりに計算されているか（速さの値そのものは見ない）"""
    j = load(f"{R}/{label}.json")
    rows = list(__import__("csv").DictReader(open(f"{R}/{label}_steps.csv")))
    rec = [r for r in rows if r["warmup"] == "False"]
    w = np.array([float(r["wait_s"]) for r in rec]); c = np.array([float(r["compute_s"]) for r in rec])
    assert len(rows) == WARMUP + j["steps"] and len(rec) == j["steps"]
    assert abs(j["samples_per_s"] - j["batch"] * j["accum"] * len(rec) / (w + c).sum()) < 1e-6 * j["samples_per_s"]
    assert abs(j["f_data"] - w.sum() / (w + c).sum()) < 1e-9 and 0 <= j["f_data"] < 1
    assert abs(j["step_ms_median"] - 1000 * np.median(w + c)) < 1e-6
    assert abs(j["amdahl_x2"] - amdahl(1 - j["f_data"], 2)) < 1e-12 and abs(j["amdahl_limit"] - 1 / j["f_data"]) < 1e-9 * j["amdahl_limit"]
    return j


FILES = ["train_toy.py", "gpu_log.sh", "prep_cache.py", "train_fast.py", "fast_table.py", "prof_toy.py", "stft_bench.py"]

In [ ]:
# 確かめ（教科書 III-5-5 の「確かめること」と確認問題のうち、設定の大きさや機械に依らず成り立つことだけ）
ok(all(os.path.exists(f) for f in FILES), f"{len(FILES)} 個のファイルを書き出した")
r = subprocess.run([sys.executable, "-c", "import train_fast"], capture_output=True, text=True)
ok(r.returncode == 0, "[手順1] python -c \"import train_fast\" がエラーなく終わる" + r.stderr[-300:])
ok(abs(amdahl(0.8, 4) - 2.5) < 1e-12 and abs(amdahl(0.8, 1e12) - 5) < 1e-9, "III-5-3-4：f = 0.8 なら 4 倍速い GPU で 2.5 倍、上限 5 倍（train_fast.py の amdahl()）")

## 1. GPU と CPU のようすを見る（III-5-3-2）

`nvidia-smi` は NVIDIA の GPU があるとき（Colab の GPU、クラウドのインスタンス）だけ使えます。見る場所は、GPU-Util（計算が 1 つでも動いていた時間の割合。「計算能力を何%使っているか」ではない）、Memory-Usage（PyTorch のキャッシュの分だけ大きく出る）、Pwr:Usage/Cap（本当に忙しいと上限近く）、Temp・Perf、Processes です。Mac には `nvidia-smi` が無いので、説明だけになります（Mac ではアクティビティモニタの GPU の履歴でおおよそを見ます）。

CPU のコア数（`nproc`）も記録します。ワーカーは CPU のコアを 1 つずつ使うので、コア数より多くしても速くなりません。Colab の CPU は 2 コア前後のことが多い（公式で要確認）ので、ワーカーで待ちを隠しにくい環境です。

In [ ]:
HAS_SMI = shutil.which("nvidia-smi") is not None and DEV.type == "cuda"   # nvidia-smi があり、学習も GPU で動くときだけ
if HAS_SMI:
    subprocess.run(["nvidia-smi"])
    subprocess.run(["nvidia-smi", "--query-gpu=name,utilization.gpu,memory.used,memory.total,power.draw,power.limit,temperature.gpu",
                    "--format=csv"])
    print("compute capability", torch.cuda.get_device_capability(), "（8 以上＝ Ampere 世代以降なら bf16 が速い）")
else:
    print("nvidia-smi が無いか、GPU を使っていません（Mac・CPU だけのランタイム）。8 節の gpu_log.sh もとばします")
print("CPU のコア数（nproc）:", NCPU)

## 2. データを保存する（手順 2、`prep_cache.py`）

`train_toy.py` の合成データを、前もって計算して `data/toy_train/` に保存します（波形 `x.npy` と、`--stft` で対数振幅スペクトログラム `stft.npy`）。学習のときは `CachedMotor` が `np.load(…, mmap_mode="r")`（memmap）で必要な行だけを読みます。

**既定は 4000 個**（約 66 MB ＋ 67 MB）。教科書の本番は 20000 個（約 330 MB ＋ 340 MB。筆者の Mac で波形 3 秒、STFT 1 秒）です。Colab のディスクは十分ありますが、ランタイムを切ると消えます。

In [ ]:
run(f"prep_cache.py --n {N_CACHE} --length 4096 --seed 0 --name toy_train --stft")
# 本番（教科書の手順2）：
# run("prep_cache.py --n 20000 --length 4096 --seed 0 --name toy_train --stft")

In [ ]:
# 確かめ（教科書 III-5-5 の「確かめること」と確認問題のうち、設定の大きさや機械に依らず成り立つことだけ）
info = load("data/toy_train/cache_info.json")
x = np.load("data/toy_train/x.npy", mmap_mode="r"); s = np.load("data/toy_train/stft.npy", mmap_mode="r"); y = np.load("data/toy_train/y.npy")
ok(x.shape == (N_CACHE, 4096) and x.dtype == np.float32 and s.dtype == np.float16 and len(y) == N_CACHE,
   f"[手順2] x.npy {x.shape} float32、stft.npy {s.shape} float16、y.npy {len(y)} 個")
ok(info["stft_shape"] == [129, 65] and s.shape[1:] == (129, 65), "STFT の形は (n_fft/2+1, 4096/hop+1) = (129, 65)（n_fft 256、hop 64、center=True）")
ok(abs(20000 * 4096 * 4 / 1e6 - 330) < 5 and abs(20000 * 129 * 65 * 2 / 1e6 - 340) < 5,
   "本番の 20000 個なら、波形 20000×4096×4 バイト ≈ 330 MB、STFT 20000×129×65×2 バイト ≈ 340 MB（III-5-2 の表）")
ok(y.sum() == N_CACHE // 2, "正常と異常が半分ずつ（SynthMotor の y = i % 2）")
from train_toy import SynthMotor
ok(np.array_equal(x[7], SynthMotor(N_CACHE, 4096, seed=0)[7][0][0].numpy()), "保存した 7 番の波形は、SynthMotor がその場で作るものと同じ（番号から決まる乱数）")
print(f"かかった時間：波形 {info['wave_s']} 秒、STFT {info['stft_s']} 秒（筆者の Mac で 20000 個なら 3 秒・1 秒）")

## 3. データ待ちを測る（手順 3、`train_fast.py` の (1)）

1 ステップ（128 サンプル）を「データ待ち」と「GPU の計算」に分けて測ります。GPU の計算は非同期なので、`train_fast.py` は時刻を読む前に `sync()`（`torch.cuda.synchronize()`／`torch.mps.synchronize()`）で GPU を待っています。

- `--data synth`：`train_toy.py` と同じく、信号を毎回 CPU で合成する（III-5-3-3 の ①）
- `--data cache`：2 節で保存したファイルを memmap で読む（合成の手間は 0 になるが、読む手間が残る）
- `--workers`：DataLoader のワーカー数。ワーカーが次のバッチを先に作り、待ちを計算の裏に隠す（隠し切れる目安は $t_{\text{data}}/n_{\text{workers}} \le t_{\text{gpu}}$）

各条件で、最初の 10 ステップ（ウォームアップ）を除いた中央値を見ます。ワーカーを使う条件は、ワーカーの起動で最初のステップが長くなります。

In [ ]:
LABELS1 = []
for w in WORKERS:
    run(f"train_fast.py --data synth --workers {w} --amp {BASE_AMP} --steps {STEPS} --outdir {R} --label synth_w{w}")
    LABELS1.append(f"synth_w{w}")
for w in (0, W_FAST):
    run(f"train_fast.py --data cache --workers {w} --amp {BASE_AMP} --steps {STEPS} --outdir {R} --label cache_w{w}")
    LABELS1.append(f"cache_w{w}")
# 本番（教科書の (1)。Mac）：STEPS = 120、ワーカー 0 2 4（synth）と 0 2（cache）、N_CACHE = 20000 のデータで
# 本番（インスタンス）：STEPS = 300、ワーカー 0 2 4 8（synth）と 0 4（cache）、--amp fp16

In [ ]:
# 確かめ（教科書 III-5-5 の「確かめること」と確認問題のうち、設定の大きさや機械に依らず成り立つことだけ）
J1 = [check_run(lab) for lab in LABELS1]   # 定義と合わなければ check_run の中で止まる
ok(len(J1) == len(LABELS1) >= 3 and all(j["steps"] == STEPS and j["samples_per_s"] > 0 and 0 <= j["f_data"] < 1 for j in J1),
   f"[手順3] {len(LABELS1)} 条件（{', '.join(LABELS1)}）で、サンプル/秒・待ちの割合・アムダールの値が、steps.csv から本文の定義どおりに計算されている")
print(f"{'label':<10}{'待ち ms':>9}{'計算 ms':>9}{'待ち%':>7}{'サンプル/秒':>12}{'初回 s':>8}")
for lab in LABELS1:
    j = load(f"{R}/{lab}.json")
    print(f"{lab:<10}{j['wait_ms_median']:>9.1f}{j['compute_ms_median']:>9.1f}{100 * j['f_data']:>7.0f}{j['samples_per_s']:>12.0f}{j['first_step_s']:>8.2f}")
print("（ワーカーを増やすと待ちが減るか、synth と cache でどう違うかは、機械で変わる。III-5-7 の表と比べる）")

**待ちと計算を図にする**（ノートブック版で足したセル）。左は条件ごとの 1 ステップの中央値（待ち＋計算）、右は `synth_w0` の待ち・計算の時系列（ウォームアップを除く。III-5-8 の発展課題1 のヒストグラムの元になるもの）です。ワーカーがあると、待ちはほとんど 0 で、ときどき大きく待つ回（ワーカーが追いつかなかった回）が出ます。

In [ ]:
import csv, logging, warnings
import matplotlib.pyplot as plt
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Glyph")
plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # 日本語のフォントは 0-1 のセルで用意する（見つからなければそこで知らせる）

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
js = [load(f"{R}/{lab}.json") for lab in LABELS1]
wm = [j["wait_ms_median"] for j in js]; cm = [j["compute_ms_median"] for j in js]
ax1.barh(LABELS1, wm, label="データ待ち", color="tab:orange")
ax1.barh(LABELS1, cm, left=wm, label="計算", color="tab:blue")
ax1.set_xlabel("1 ステップ（中央値、ms）"); ax1.invert_yaxis(); ax1.legend(); ax1.set_title(f"{DEV.type}、AMP {BASE_AMP}")
for lab in ("synth_w0", f"synth_w{W_FAST}" if W_FAST in WORKERS else LABELS1[1]):
    rows = [r for r in csv.DictReader(open(f"{R}/{lab}_steps.csv")) if r["warmup"] == "False"]
    ax2.plot([1e3 * float(r["wait_s"]) for r in rows], ".-", label=f"{lab} の待ち")
ax2.plot([1e3 * float(r["compute_s"]) for r in rows], ".-", color="gray", alpha=0.6, label=f"{lab} の計算")
ax2.set_xlabel("ステップ（ウォームアップの後）"); ax2.set_ylabel("ms"); ax2.legend(); ax2.set_title("ステップごとの時間")
fig.tight_layout(); plt.show()

## 4. アムダールの法則で上限を出す（手順 4、III-5-3-4）

`synth_w0` の中央値から、GPU で速くなる部分の割合 $f = t_{\text{計算}}/(t_{\text{待ち}} + t_{\text{計算}})$ を出し、計算が 10 倍速い GPU に替えたときの倍率

$$A(10) = \frac{1}{(1-f) + f/10},\qquad 上限 \ \frac{1}{1-f}$$

と、待ちを隠し切るのに要るワーカーの数 $n \ge t_{\text{data}}/(t_{\text{gpu}}/10)$ を計算します（データ待ちの速さは変わらないと仮定）。教科書の例（Apple シリコン の実測：待ち 19.2 ms、計算 51.1 ms）と並べます。Colab の GPU で動かしているときは、すでに計算が速い GPU の上の値なので、$f$ が Mac より小さく出るはずです（III-5-7 の「クラウドでは synth_w0 の待ちの割合は Mac よりずっと大きくなるはず」）。

In [ ]:
def amdahl_report(name, t_wait, t_gpu, S=10):
    f = t_gpu / (t_wait + t_gpu)
    A, lim = amdahl(f, S), 1 / (1 - f)
    n_need = t_wait / (t_gpu / S)
    print(f"{name}：待ち {t_wait:.1f} ms、計算 {t_gpu:.1f} ms → f = {f:.3f}、A({S}) = {A:.2f} 倍、上限 {lim:.1f} 倍、"
          f"計算が {S} 倍速いとき待ちを隠すワーカー {n_need:.1f} 個以上")
    return f, A, lim, n_need


j0 = load(f"{R}/synth_w0.json")
mine = amdahl_report(f"このマシン（{DEV.type}）の synth_w0", j0["wait_ms_median"], j0["compute_ms_median"])
book = amdahl_report("教科書の例（Apple シリコン、MPS）", 19.2, 51.1)
print(f"このマシンの待ちの割合：中央値から {1 - mine[0]:.0%}、合計の比（json の f_data）から {j0['f_data']:.0%}"
      "（ときどき大きく待つ回があると、合計の比のほうが大きく出る）")

In [ ]:
# 確かめ（教科書 III-5-5 の「確かめること」と確認問題のうち、設定の大きさや機械に依らず成り立つことだけ）
f, A, lim, n_need = book
ok(abs(f - 0.727) < 0.001 and abs(A - 2.89) < 0.01 and abs(lim - 3.7) < 0.05 and abs(n_need - 3.8) < 0.05,
   f"III-5-3-4 の例：f = 51.1/70.3 = {f:.3f}、A(10) = {A:.2f}、上限 {lim:.1f} 倍、ワーカー {n_need:.1f} → 4 個以上")
t24 = 19.2 + 51.1 / 10
ok(abs(t24 - 24.3) < 0.05 and abs((51.1 / 10) / t24 - 0.21) < 0.005, f"ワーカー 0 のまま：1 ステップ {t24:.1f} ms のうち計算 5.1 ms で、GPU-Util はおよそ 21%（η ≈ 0.21）")
ok(abs(70.3 / 5.11 - 13.8) < 0.05, "ワーカーで待ちを隠し切れれば 1 ステップ ≈ 5.1 ms で、約 13.8 倍")
ok(abs(mine[1] - amdahl(mine[0], 10)) < 1e-12 and mine[1] <= mine[2], "このマシンの値でも A(10) ≤ 上限 1/(1−f)")

## 5. 計算を速くする工夫（手順 5、`train_fast.py` の (2) と `fast_table.py`）

データ待ちを消した `cache_w2`（保存したデータ、ワーカー 2）を基準に、1 つずつ変えます。

| 工夫 | 何をするか | 見るところ |
|---|---|---|
| AMP（`--amp fp16`・`bf16`） | 行列の掛け算・畳み込みを 16 ビットで計算。fp16 は CUDA では GradScaler を自動で使う | 計算 ms。CUDA の Tensor コアでは速くなる。MPS ではかえって遅いことがある（教科書：約 7%） |
| 勾配累積（`--batch 64 --accum 2`） | 64 個で 2 回逆伝播して勾配をため、1 回更新（実効 128） | 速くはならない（計算量は同じ） |
| `torch.compile`（`--compile`） | 小さな演算をまとめた速いコードを作る | 計算 ms が減る代わりに、初回 s が長くなる |

このマシンで試す AMP は次のとおりです：`cuda` では基本が fp16 なので、比べる相手として `off` と（Ampere 世代以降なら）`bf16`。`mps` では教科書どおり fp16 と bf16。`cpu` では bf16 だけ（`train_fast.py` は CPU の fp16 を止めます）。T4 は bf16 に対応していない世代なので、`bf16` はとばします（III-5-7 の表の「--amp bf16 でエラーや極端に遅い」）。

`torch.compile` は、初回のコンパイルに数十秒〜数分かかることがあります。C コンパイラが無い、版の組み合わせが合わないなどでエラーになったら、そのセルは「とばした」と表示して先へ進みます（教科書：外して動けば、この章の目的には十分）。

In [ ]:
base = f"cache_w{W_FAST}"
if DEV.type == "cuda":
    major = torch.cuda.get_device_capability()[0]
    AMPS = ["off"] + (["bf16"] if major >= 8 else [])
elif DEV.type == "mps":
    AMPS = ["fp16", "bf16"]
else:
    AMPS = ["bf16"]
LABELS2 = [base]
for amp in AMPS:
    run(f"train_fast.py --data cache --workers {W_FAST} --amp {amp} --steps {STEPS} --outdir {R} --label {base}_{amp}")
    LABELS2.append(f"{base}_{amp}")
run(f"train_fast.py --data cache --workers {W_FAST} --amp {BASE_AMP} --batch 64 --accum 2 --steps {STEPS} --outdir {R} --label {base}_acc2")
LABELS2.append(f"{base}_acc2")
# 本番（教科書の (2)）：STEPS = 120（インスタンスでは 300、ワーカー 4）

In [ ]:
try:
    run(f"train_fast.py --data cache --workers {W_FAST} --amp {BASE_AMP} --compile --steps {STEPS} --outdir {R} --label {base}_compile")
    LABELS2.append(f"{base}_compile")
except AssertionError as e:
    print("torch.compile はこの環境では動かなかったので、とばします（III-5-7 の表の「--compile で長い時間止まる、エラー」）：", e)

In [ ]:
out = run(f"fast_table.py --dir {R} --base {base}")
# 教科書：Mac は fast_table.py（基準 synth_w0）、インスタンスは fast_table.py --dir "$R" --base synth_w0

In [ ]:
# 確かめ（教科書 III-5-5 の「確かめること」と確認問題のうち、設定の大きさや機械に依らず成り立つことだけ）
J2 = [check_run(lab) for lab in LABELS2]   # 定義と合わなければ check_run の中で止まる
ok(len(J2) == len(LABELS2) >= 1 and all(j["steps"] == STEPS and j["samples_per_s"] > 0 and 0 <= j["f_data"] < 1 for j in J2),
   f"[手順5] {len(LABELS2)} 条件（{', '.join(LABELS2)}）の結果が本文の定義どおり")
lines = out.splitlines()
head = next(i for i, l in enumerate(lines) if l.startswith("label"))
body = lines[head + 1:head + 1 + len(set(LABELS1 + LABELS2))]
ok(len(body) == len(set(LABELS1 + LABELS2)) and all(len(l.split()) == 13 for l in body),
   f"表は {len(body)} 行（{R} の train_fast.py の結果すべて）で、どの行も 13 列そろっている")
ok(any(l.split()[0] == base and l.split()[7] == "1.00" for l in body), f"基準（{base}）の倍率は 1.00")
acc = load(f"{R}/{base}_acc2.json")
ok(acc["batch"] * acc["accum"] == 128, "勾配累積 64×2 の実効バッチは 128（基準と同じ。サンプル/秒も 128 サンプルを 1 ステップとして数える）")
jb = load(f"{R}/{base}.json")
for lab in LABELS2[1:]:
    j = load(f"{R}/{lab}.json")
    print(f"{lab:<20}計算 {j['compute_ms_median']:6.1f} ms（基準 {jb['compute_ms_median']:.1f}）、初回 {j['first_step_s']:5.2f} s（基準 {jb['first_step_s']:.2f}）、"
          f"メモリ {j['peak_mem_mb']:.0f} MB、損失 {j['final_loss']:.3f}")
print("（compile で計算が短くなり初回が長くなるか、勾配累積で速くならないか、を見る。MPS のメモリの値は最大値ではない）")

**torch.compile の損得**（図III-5-5 と同じ考え方。ノートブック版で足したセル）。合計時間を「最初のステップ ＋ 1 ステップの中央値 ×（N − 1）」で近似し、compile が元を取るステップ数 $N^{*}$ を出します。

In [ ]:
if f"{base}_compile" in LABELS2:
    jc = load(f"{R}/{base}_compile.json")
    gain = (jb["step_ms_median"] - jc["step_ms_median"]) / 1e3            # 1 ステップの節約（秒）
    extra = jc["first_step_s"] - jb["first_step_s"]                     # 初回の余分（秒）
    if gain > 0:
        print(f"1 ステップ {1e3 * gain:.1f} ms の節約で、初回の余分 {extra:.2f} 秒を取り返すのは約 {1 + extra / gain:.0f} ステップ目")
    else:
        print("このマシンでは compile で 1 ステップが短くならなかった（外したままでよい）")
else:
    print("compile をとばしたので計算しません")

**バッチサイズと VRAM**（III-5-3-6 の $M(B) \approx M_0 + m_1 B$。`cuda` のときだけ。ノートブック版で足したセル）。`torch.cuda.max_memory_allocated()`（`peak_mem_mb`）をバッチ 32・64・128 で測り、2 点以上から $M_0$ と $m_1$ を最小二乗で求めます（III-5-8 の発展課題4 の小さい版）。MPS の値（`driver_allocated_memory()`）は最大値ではないので、`cuda` 以外ではとばします。

In [ ]:
if DEV.type == "cuda":
    Bs, Ms = [32, 64, 128], []
    for b in Bs:
        run(f"train_fast.py --data cache --workers 0 --amp {BASE_AMP} --batch {b} --steps 5 --warmup 2 --outdir {R}/mem --label mem_b{b}")
        Ms.append(load(f"{R}/mem/mem_b{b}.json")["peak_mem_mb"])
    m1, M0 = np.polyfit(Bs, Ms, 1)
    vram = torch.cuda.get_device_properties(0).total_memory / 2**20
    print(f"M(B) ≈ {M0:.0f} MB + {m1:.2f} MB × B → VRAM {vram:.0f} MB から 1 GB 残すと、B は {(vram - 1024 - M0) / m1:.0f} くらいまで（2 点以上の直線の外挿。実際は 1 回試す）")
else:
    print("cuda ではないのでとばします（教科書の確認問題6 の計算は 9 節で確かめます）")

## 6. 前処理の置き場所を比べる（手順 6、`stft_bench.py`）

1 バッチ（128 本 × 4096 点）の前処理を、次の置き場所で測ります（III-5-3-5 の A・B・C）。

- **A. CPU で毎回**：STFT・CWT を CPU で
- **B. GPU で毎回**：STFT（転送の往復込みと、GPU に載せたまま）、CWT（48 周波数の複素モルレーを並べたフィルタで `conv1d`）
- **C. 保存して読むだけ**：2 節の `stft.npy`・`x.npy` から 128 個を memmap で読む

CUDA では CWT（長いフィルタの畳み込み）が GPU で速くなることが多く、教科書の Mac（MPS）では逆に CPU より遅くなりました。`device` が `cpu` のときは、「GPU」の欄も CPU で計算した値です。

**既定は各 10 回**（教科書は `--reps 30`）。

In [ ]:
out6 = run(f"stft_bench.py --cache-dir data/toy_train --reps 10 --out {R}/stft_bench.json")
# 本番（教科書の (3)）：
# run(f"stft_bench.py --cache-dir data/toy_train --out {R}/stft_bench.json")

In [ ]:
# 確かめ（教科書 III-5-5 の「確かめること」と確認問題のうち、設定の大きさや機械に依らず成り立つことだけ）
j = load(f"{R}/stft_bench.json")
ok(j["stft_max_rel_err"] < 1e-5, f"[手順6] STFT の CPU と {j['device']} の最大相対誤差が 1e-5 より小さい（{j['stft_max_rel_err']:.1e}。同じ計算をしている）")
ok(j["cwt_max_rel_err"] < 1e-4, f"CWT の CPU と {j['device']} の最大相対誤差 {j['cwt_max_rel_err']:.1e}（float32 の長い畳み込みの和なので、目安を 1e-4 にした）")
ok(j["cwt_taps"] == 1835 and j["cwt_shape"] == [48, 64] and j["stft_shape"] == [129, 65],
   "III-5-3-5：f_k = 50 Hz、n_c = 6、±3σ のフィルタは 1835 点。CWT は (48 周波数, 64)、STFT は (129, 65)")
half = math.ceil(3 * 6 / (2 * math.pi * 50) * 16000)
ok(2 * half + 1 == 1835, f"1835 = 2 × ceil(3 σ_k f_s) + 1（σ_k = n_c/(2π f_k) = {6 / (2 * math.pi * 50) * 1e3:.2f} ms、半分の長さ {half} 点）")
jb = load(f"{R}/cache_w{W_FAST}.json")
print(f"1 バッチの前処理（STFT、{j['device']} に載せたまま {j['stft_gpu_only_ms']:.2f} ms、CPU {j['stft_cpu_ms']:.2f} ms）と、"
      f"学習の 1 ステップの計算 {jb['compute_ms_median']:.1f} ms を比べる（前処理が十分小さく、ワーカーで隠れるなら、事前計算は要らない）")

## 7. プロファイラで 1 ステップの中身を見る（手順 7、`prof_toy.py`）

`torch.profiler` で 6 ステップを記録し、演算ごとの時間を表にします。表の上位に、データ待ち（`enumerate(DataLoader)…`。`prof_toy.py` の `record_function("data_wait")` の中身）と計算（`train_step`）が出ます。CUDA では GPU の時間（`Self CUDA`）で並べ、Chrome のトレース（`prof_cuda_trace.json`。`chrome://tracing` や https://ui.perfetto.dev で開く）も保存します。MPS・CPU では CPU 側の時間で並べます。

In [ ]:
out7 = run(f"prof_toy.py --workers 0 --outdir {R}")

In [ ]:
# 確かめ（教科書 III-5-5 の「確かめること」と確認問題のうち、設定の大きさや機械に依らず成り立つことだけ）
ok(os.path.exists(f"{R}/prof_{DEV.type}.txt") and "Name" in out7, f"[手順7] プロファイラの表が {R}/prof_{DEV.type}.txt に保存された")
DATA_ROW = "enumerate(DataLoader)" if "enumerate(DataLoader)" in out7 else "data_wait"
print("表に train_step と、データ待ちの行（" + DATA_ROW + "）が", "ある" if ("train_step" in out7 and DATA_ROW in out7) else "無い（上位 15 行に入らなかった。CUDA では GPU の時間で並べるので出ないことがある）")
if DEV.type == "cuda":
    ok(os.path.exists(f"{R}/prof_cuda_trace.json"), "CUDA なので Chrome のトレースも保存された")


def cpu_total_ms(name):
    """表の行から CPU total（単位つき）を ms で読む"""
    hdr = next(l for l in out7.splitlines() if "CPU total" in l and "Name" in l)
    cols = [c for c in hdr.split("  ") if c.strip()]
    row = next(l for l in out7.splitlines() if l.strip().startswith(name))
    vals = [v for v in row.split("  ") if v.strip()]
    v = vals[[c.strip() for c in cols].index("CPU total")].strip()
    for u, k in (("ms", 1), ("us", 1e-3), ("s", 1e3)):
        if v.endswith(u):
            return float(v[:-len(u)]) * k


try:
    a, b = cpu_total_ms("train_step"), cpu_total_ms(DATA_ROW)
    print(f"CPU total：train_step {a:.1f} ms、{DATA_ROW} {b:.1f} ms → 比 {a / b:.1f}"
          f"（3 節の synth_w0 の 計算 : 待ち ＝ {j0['compute_ms_median'] / j0['wait_ms_median']:.1f}。教科書の Mac では 2.6 と 2.7）")
except Exception as e:
    print("表の形が版で違うので比は計算しませんでした。上の表の CPU total を見てください", e)

## 8. `gpu_log.sh` で GPU-Util を記録する（手順 8、`nvidia-smi` があるときだけ）

教科書の手順 8 では、Vast.ai のインスタンスで `bash gpu_log.sh synth_w0 -- uv run --locked python train_fast.py …` のように、学習の間ずっと `nvidia-smi --query-gpu=… -l 1` の値を CSV に記録し、終わったら GPU-Util の平均などを表示します。Colab の GPU でも同じことができます（記録は 1 秒ごとなので、学習が数秒で終わると行が少なくなります。ここでは 300 ステップにしています）。

- 確かめること：`synth_w0` の GPU-Util の平均が低く（待ちの割合が大きく）、ワーカーを増やした `synth_w2` や保存したデータの `cache_w2` で上がる。
- Colab はセルの実行中に別のセルを動かせないので、教科書の「別の端末で `nvidia-smi dmon -s pucm -d 1`」は、ターミナル（Colab の有料のプランの機能。公式で要確認）があればそこで実行します。
- `nvidia-smi` が無いとき（Mac、CPU だけのランタイム）は、このセルは何もしません（`gpu_log.sh` も、`nvidia-smi` が無ければ記録せずにコマンドだけ実行する作りです）。

In [ ]:
GPU_STEPS = 300   # 教科書のインスタンスと同じ
if HAS_SMI:
    os.environ["GPU_LOG_DIR"] = os.path.abspath("runs/gpu_log")
    for lab, opt in [("synth_w0", "--data synth --workers 0"), (f"synth_w{W_FAST}", f"--data synth --workers {W_FAST}"),
                     (f"cache_w{W_FAST}", f"--data cache --workers {W_FAST}")]:
        run(f"train_fast.py {opt} --amp {BASE_AMP} --steps {GPU_STEPS} --outdir {R}/gpu_log_runs --label {lab}",
            prefix=("bash", "gpu_log.sh", lab, "--"))
    print(sorted(os.listdir("runs/gpu_log")))
else:
    print("nvidia-smi が無い（または device が cuda でない）ので、gpu_log.sh の記録はとばします。Colab の GPU で実行すると、ここで GPU-Util の平均が表示されます")

**III-3 の $\eta$ を説明する**（手順 10）。III-3 のベンチは計算だけを測っているので、精度をそろえていれば $\eta \approx t_{\text{gpu}}/(t_{\text{wait}} + t_{\text{gpu}}) = 1 - (\text{データ待ちの割合})$ です。下のセルは、3 節の結果から各条件の $1-(\text{待ちの割合})$ を表示します。`cuda` で `--amp fp16`（`train_toy.py` と同じ精度）の行が、III-3 で求めた $\eta$ と比べる値です。$\eta$ が低いマシンは、GPU ではなく CPU が足りないと読みます。

In [ ]:
for lab in LABELS1:
    j = load(f"{R}/{lab}.json")
    print(f"{lab:<10} 1 − 待ちの割合 = {1 - j['f_data']:.2f}（AMP {j['amp']}、{j['device_name']}）")

## 9. 確認問題の数値を確かめる

教科書 III-5 の確認問題のうち、数値の答えがあるもの（1・2・3・6）を、式で確かめます。

In [ ]:
# 確かめ（教科書 III-5-5 の「確かめること」と確認問題のうち、設定の大きさや機械に依らず成り立つことだけ）
# 1（アムダール）：待ち 10 ms、計算 30 ms、計算が 3 倍速い GPU
ok(abs((10 + 30 / 3) - 20) < 1e-12 and abs(40 / 20 - amdahl(0.75, 3)) < 1e-12 and abs(1 / (1 - 0.75) - 4) < 1e-12,
   "確認問題1：1 ステップ 20 ms、2.0 倍（A(3) = 1/(0.25 + 0.25)）、上限 4 倍")
# 2（ワーカーの数）：1 バッチ 24 ms、GPU 5 ms、4 コア
ok(math.ceil(24 / 5) == 5 and abs(24 / 4 - 6) < 1e-12 and round(5 / 6, 2) == 0.83, "確認問題2：ワーカー 5 個（n ≥ 4.8）、4 コアなら 1 ステップ約 6 ms（GPU は 83%）")
# 3（効率 η）：ベンチ 6 ms、train_toy.py 15 ms
eta = 6 / 15
ok(abs(eta - 0.4) < 1e-12 and abs((1 - eta) - 0.6) < 1e-12 and abs(1 / eta - 2.5) < 1e-12, "確認問題3：η = 0.4、待ちの割合 0.6、本番 1 時間の見積もりは実際 2.5 時間")
# 6（VRAM）：B = 32 で 1.6 GB、B = 64 で 2.8 GB、12 GB の GPU で 1 GB 残す
m1 = (2.8 - 1.6) / 32; M0 = 1.6 - 32 * m1
Bmax = (11 - M0) / m1; B2 = 2 ** int(math.log2(Bmax))
ok(abs(m1 - 0.0375) < 1e-12 and abs(M0 - 0.4) < 1e-12 and int(Bmax) == 282 and B2 == 256 and abs(M0 + m1 * B2 - 10.0) < 1e-9,
   f"確認問題6：m1 = 0.0375 GB、M0 = 0.4 GB、B ≤ {Bmax:.1f} → 2 のべき乗で 256（M = 10.0 GB）")
# III-5-3-6：AdamW の重み 1 個あたり 16 バイト、AudioCNN の重み 287,586 個で約 4.6 MB
from train_toy import AudioCNN
npar = sum(p.numel() for p in AudioCNN().parameters())
ok(npar == 287586 and abs(npar * 16 / 1e6 - 4.6) < 0.05, f"AudioCNN の重みは {npar:,} 個、重み・勾配・AdamW の状態で {npar * 16 / 1e6:.1f} MB（III-5-3-6）")
# 確認問題7：fp16 の最小の正の数（非正規化数）と最大値
ok(abs(float(np.finfo(np.float16).smallest_subnormal) - 5.96e-8) < 1e-10 and float(np.finfo(np.float16).max) == 65504,
   "確認問題7：fp16 の最小の正の数 約 6×10⁻⁸、最大 65504（bf16 は指数部が float32 と同じ 8 ビット）")
print(f"ok() の数：{N_OK}")

**結果を持ち帰る**（教科書の手順 9 の `vast_sync.sh pull fast_cuda` の代わり）。Colab のファイルはランタイムを切ると消えるので、`runs/` を zip にしてダウンロードします（Mac で動かしているときは何もしません）。Mac では `~/home-cloud/runs/` の下に展開し、`uv run --locked python fast_table.py --dir runs/fast_cuda` で表にします。`data/toy_train/` はいつでも作り直せるので持ち帰りません。

終わったら、Colab のランタイムを解放します（「ランタイム」メニューの「ランタイムを接続解除して削除」など。項目の名前は公式ドキュメントで要確認。III-2 のセル7 の `runtime.unassign()` でもかまいません）。

In [ ]:
if IN_COLAB:
    zip_path = shutil.make_archive("p3c5_runs", "zip", root_dir=".", base_dir="runs")
    from google.colab import files
    files.download(zip_path)
    # from google.colab import runtime; runtime.unassign()   # 結果をダウンロードしてから。ランタイムを返す（III-2 のセル7）
else:
    print(f"Colab ではないので、持ち帰りはしません。結果は {os.path.abspath(R)} にあります")
print(sorted(os.listdir(R)))

## おわりに

- 1 ステップを「データ待ち」と「計算」に分けて測り、アムダールの法則で「GPU を速くしても何倍までか」を数字にしました。GPU-Util が低いときは、GPU を替える前に待ちの原因（合成・前処理・読み込み・ワーカー数・CPU のコア数）を切り分けます。
- 本番の設定（`N_CACHE = 20000`、`STEPS = 120`、Colab の GPU では `STEPS = 300`）にするときは、0 節の設定のセルを書き換えて最初から実行し直してください。
- Colab の CPU はコアが少ないので、`synth` のワーカーを増やしても待ちが消えにくいはずです。そのときは保存したデータ（`cache`）や、STFT を GPU で計算する（6 節の B）が効きます。Mac とクラウドの表を並べ、「どちらでどの工夫が効いたか」を 1〜2 文でノートに書いてください。